# Quick SLM — 04a · SFT seed-pool audit

Run this before the generation cells of notebook 04. It verifies, on CPU alone, that the
unpaired SFT categories (single_stage, multi_stage, traps, refusals) plan many more examples
per (domain, subtype) cell than their seed pools can realise. The teacher then repeats itself,
and the dedup pass in notebook 04 deletes the surplus at full teacher price.

`training/SFT_AUDIT.md` raised this as open question B, but measured it at 15 seeds per pool
and before the domain/subtype cycle bug was fixed. The pools have since grown to 29 and the
cycle is fixed, so those numbers are stale. This notebook re-measures against the current code
and projects what a per-seed cap would save.

Sections 1 to 5 are CPU-only and load no teacher. Section 6 is an optional, bounded GPU
diagnostic, off by default, honouring the rule that no teacher time is spent until the fix lands.

## 1 · Framework

Same install as notebook 04. No GPU is needed for sections 1 to 5.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

EXTRAS = 'sft'
DRIVE_ROOT = '/content/drive/MyDrive/quick-slm'

import subprocess, sys, importlib
from pathlib import Path

root = Path(DRIVE_ROOT)
candidates = [root / 'code', root, root / 'quick-slm']
REPO_DIR = next((p for p in candidates if (p / 'pyproject.toml').exists()), None)
if REPO_DIR is None:
    raise RuntimeError('No pyproject.toml on Drive under ' + str(root / 'code') + '; upload the repo there and re-run.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR) + '[' + EXTRAS + ']'], check=True)

framework_dir = REPO_DIR / 'framework'
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))
importlib.invalidate_caches()

import v1.quick_slm_trainer as q
print('quick-slm-trainer', q.__version__, 'from', Path(q.__file__).parent)
# Refuse to run outside the support window this training version declares in
# training/v1/framework.json. A framework newer than v1's window would not fail
# loudly, it would build a different corpus and the difference would surface as
# unexplained numbers weeks later. If this raises, install the archived build it
# names rather than editing this cell.
if hasattr(q, 'require_framework'):
    q.require_framework('v1', REPO_DIR)
else:
    raise RuntimeError(
        'quick-slm-trainer ' + q.__version__ + ' predates the support-window check; '
        'training v1 requires >=1.0. See SUPPORT.md.'
    )


## 2 · The plan, per (domain, subtype) cell

Build the exact plan notebook 04 generates, then count how many requests land in each
unpaired cell and compare that to the number of distinct seeds the planner can draw from
(`usable_seeds`, which also narrows `optional_args` to the seeds naming a tool with an
optional parameter). `reuse` is planned / seeds: how many times, on average, each seed is
re-asked. `drawn` is how many distinct seeds actually appear in the plan, and it should equal
`seeds`, confirming the pool is exhausted and then hammered.

In [ ]:
import collections
from v1.quick_slm_trainer import sft_v1
from v1.quick_slm_trainer.sft import plan_requests, describe_plan
from v1.quick_slm_trainer.sft.specs import CATEGORIES
from v1.quick_slm_trainer.sft.prompts import usable_seeds

cfg = sft_v1()
requests = plan_requests(cfg.sft)
print(describe_plan(requests))
print()

unpaired = [k for k, s in CATEGORIES.items() if not s.paired]
cell_reqs = collections.defaultdict(list)
for r in requests:
    if not r.is_paired:
        cell_reqs[(r.category, r.domain, r.subtype)].append(r)

header = 'category'.ljust(16) + 'domain'.ljust(9) + 'subtype'.ljust(22) + 'planned'.rjust(9) + 'seeds'.rjust(7) + 'drawn'.rjust(7) + 'reuse'.rjust(9)
print(header)
print('-' * len(header))

base = 0
for cat in unpaired:
    spec = CATEGORIES[cat]
    for domain in spec.domains:
        for subtype in spec.subtypes_for(domain):
            reqs = cell_reqs.get((cat, domain, subtype), [])
            planned = len(reqs)
            nseeds = len(usable_seeds(cat, domain, subtype))
            drawn = len({r.seed_topic for r in reqs})
            reuse = planned / nseeds if nseeds else float('inf')
            base += planned
            row = cat.ljust(16) + domain.ljust(9) + subtype.ljust(22) + f'{planned:>9,}' + f'{nseeds:>7}' + f'{drawn:>7}' + f'{reuse:>8.0f}' + 'x'
            print(row)

print('-' * len(header))
topics = {r.seed_topic for r in requests if not r.is_paired}
print(f'{base:,} unpaired examples planned from {len(topics)} distinct seed strings ({base/len(topics):.0f}x average reuse)')

## 3 · Why the reuse turns into duplicates

Within one cell the teacher has only three sources of variety: the seed topic (capped by the
pool just measured), the sampled tool list, and sampling at temperature 0.7. The dedup
fingerprint in notebook 04 is the user request plus the call signatures, with the think block
excluded. For the `direct` subtype the guidance pins the phrasing (state it plainly, naming
the thing the tool needs) and the call is the seed's own required tool, so both fingerprint
axes are nearly fixed per seed and dedup collapses that cell hardest. `paraphrased` and
`irrelevant_context` keep more surface variety and survive better.

The cell below shows the secondary axis, the sampled tool list, does not rescue `direct`: the
tool the example actually calls is the seed's required tool, so the distinct-call count per
cell is bounded by the required-tool sets, not by the tool lists.

In [ ]:
print('single_stage: within-cell variety beyond the seed topic')
print()
sub_header = 'domain'.ljust(9) + 'subtype'.ljust(20) + 'requests'.rjust(10) + 'tool-lists'.rjust(12) + 'required-tool sets'.rjust(20)
print(sub_header)
print('-' * len(sub_header))
for domain in CATEGORIES['single_stage'].domains:
    for subtype in CATEGORIES['single_stage'].subtypes_for(domain):
        reqs = cell_reqs[('single_stage', domain, subtype)]
        toollists = {tuple(sorted(t['name'] for t in r.tools)) for r in reqs}
        requires = {r.requires for r in reqs}
        print(domain.ljust(9) + subtype.ljust(20) + f'{len(reqs):>10,}' + f'{len(toollists):>12,}' + f'{len(requires):>20}')
print()
print('The tool list varies, but the tool the example calls is pinned to the seed, so the number')
print('of distinct calls per cell is bounded by the required-tool sets column. For direct, the')
print('user phrasing is pinned too, which is why dedup deletes that cell hardest.')

## 4 · Contrast: the paired category is already capped

`state_memory_conflict` plans in counterfactual pairs, and its plan is capped at
`recommended_conflict_pairs()`, the point past which extra pairs are duplicates the dedup
pass deletes. The unpaired categories have no equivalent cap. That asymmetry is the bug.

In [ ]:
from v1.quick_slm_trainer.sft.conflict import recommended_conflict_pairs, train_capacity

paired = [r for r in requests if r.is_paired]
print('paired category: state_memory_conflict')
print('  train_capacity()             =', train_capacity())
print('  recommended_conflict_pairs() =', recommended_conflict_pairs())
print('  cfg.sft.conflict_max_pairs   =', cfg.sft.conflict_max_pairs)
print('  planned branches             =', len(paired), 'that is', len(paired) // 2, 'pairs')
print()
print('The paired plan is held at the distinguishable ceiling. The unpaired categories are not.')

## 5 · What a per-seed cap would save (projection)

A per-seed cap C limits each cell to min(planned, C * seeds) requests, exactly as
`conflict_max_pairs` caps the paired plan. This is arithmetic on the existing plan, changing
no source; it shows the teacher time a cap would return before we implement it. Read it
against the realised yield in section 6: the cap should sit near the yield, not below it.

In [ ]:
def project(cap_per_seed):
    total = 0
    for cat in unpaired:
        spec = CATEGORIES[cat]
        for domain in spec.domains:
            for subtype in spec.subtypes_for(domain):
                reqs = cell_reqs.get((cat, domain, subtype), [])
                nseeds = len(usable_seeds(cat, domain, subtype))
                total += min(len(reqs), cap_per_seed * nseeds)
    return total

print('cap/seed'.ljust(10) + 'planned'.rjust(10) + 'of current'.rjust(12) + 'saved'.rjust(9))
print('-' * 41)
for cap in (20, 30, 40, 60, 100):
    p = project(cap)
    print(str(cap).ljust(10) + f'{p:>10,}' + f'{100*p/base:>11.0f}%' + f'{100*(base-p)/base:>8.0f}%')
print()
print(f'current unpaired planned: {base:,}')

## 6 · Optional GPU diagnostic: realised distinct-yield per seed

Sections 1 to 5 bound the waste from the plan alone. This cell measures what the teacher
actually yields per seed on a small, bounded sample, which is the empirical form of the
deletes-about-half claim and a preview of the distinct-turns-per-seed measurement the fix
adds to the pre-flight. It is off by default: set `RUN_TEACHER_DIAGNOSTIC = True` only when
you accept spending a few hundred generations. It is not the 80k run.

The fingerprint here (user text plus call names) is a light stand-in for the full dedup
fingerprint, faithful for `direct` because its call is fixed per seed. The number it prints,
distinct-yield per seed, is the value a per-seed cap should be set to.

In [ ]:
RUN_TEACHER_DIAGNOSTIC = False   # keep False until the fix lands
DIAG_CELL = ('single_stage', 'world', 'direct')   # the subtype dedup hits hardest
DIAG_PER_SEED = 20               # completions per seed; a few hundred generations total

if not RUN_TEACHER_DIAGNOSTIC:
    print('RUN_TEACHER_DIAGNOSTIC is False, so no teacher is loaded.')
    print('Sections 1 to 5 already establish the structural waste with no GPU.')
    print('Set it True only to confirm the realised yield; it is a preview of the')
    print('distinct-turns-per-seed measurement the fix adds to the pre-flight.')
else:
    import dataclasses
    from v1.quick_slm_trainer.sft import generate as G
    from v1.quick_slm_trainer.sft.validate import extract_json
    from v1.quick_slm_trainer.sft.dedup import dedup_indices

    cat, domain, subtype = DIAG_CELL
    template = next(r for r in requests if (r.category, r.domain, r.subtype) == DIAG_CELL)
    pool = usable_seeds(cat, domain, subtype)

    diag = []
    for s in pool:
        for j in range(DIAG_PER_SEED):
            diag.append(dataclasses.replace(template, id='diag-' + str(len(diag)), seed_topic=s.topic, requires=s.requires))

    teacher, tok = G.load_teacher(cfg.sft)
    texts = []
    step = cfg.sft.gen_batch_size
    for i in range(0, len(diag), step):
        batch = diag[i:i + step]
        texts.extend(G.generate_batch(teacher, tok, cfg.sft, batch, seed=G.batch_seed(20240201, batch)))

    fps = []
    for raw in texts:
        obj = extract_json(raw)
        if obj is None:
            continue
        names = []
        for turn in obj.get('turns', []):
            for c in (turn.get('calls') or []):
                names.append(str(c.get('name', '')))
        fps.append(str(obj.get('user', '')) + ' || ' + ' '.join(names))

    kept = dedup_indices(fps, threshold=cfg.sft.dedup_jaccard, num_perm=cfg.sft.minhash_perms)
    parsed = len(fps)
    survivors = len(kept)
    print('cell', DIAG_CELL)
    print('  generated           :', len(diag))
    print('  parsed as JSON      :', parsed)
    print('  distinct after dedup:', survivors)
    print('  yield per seed      :', round(survivors / len(pool), 1), 'from', len(pool), 'seeds')
    if parsed:
        print('  dedup deletion      :', str(round(100 * (parsed - survivors) / parsed)) + '%')